In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/english-to-hindi-parallel-dataset/newdata.csv


In [8]:
import torch
import torch.nn as nn

In [9]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers = 1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first = True)
    
    def forward(self, X):
        emb = self.emb(X)
        output, (hidden, cell) = self.lstm(emb)

        return hidden, cell

In [10]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers = 1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first = True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, X, hidden, cell):
        X = X.unsqueeze(1)
        emb = self.emb(X)
        output, (hidden, cell) = self.lstm(emb, (hidden, cell))
        
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

In [11]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):
        batch_size, trg_len = trg.shape
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, vocab_size)

        hidden, cell = self.encoder(src)

        X = trg[:, 0]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(X, hidden, cell)
            outputs[:, t, :] = output
            X = trg[:, t]           

        return outputs

In [12]:
df = pd.read_csv(r'/kaggle/input/english-to-hindi-parallel-dataset/newdata.csv')
df.head()

,Unnamed: 0,english_sentence,hindi_sentence
0,0,politicians do not have permission to do what ...,"राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह कर..."
1,1,"I'd like to tell you about one such child,",मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहू...
2,2,This percentage is even greater than the perce...,यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।
3,3,what we really mean is that they're bad at not...,हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते
4,4,.The ending portion of these Vedas is called U...,इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।


In [13]:
df = df.drop('Unnamed: 0', axis = 1)

In [14]:
df.dropna(inplace = True)

In [15]:
english = df['english_sentence']
hindi = df['hindi_sentence']

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
hin_tokenizer = Tokenizer(oov_token = "<nothing>")
eng_tokenizer = Tokenizer(oov_token = "<nothing>")

2026-02-27 13:32:26.815708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772199147.044554    1445 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772199147.107208    1445 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772199147.634629    1445 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772199147.634676    1445 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772199147.634678    1445 computation_placer.cc:177] computation placer alr

In [17]:
hin_tokenizer.fit_on_texts(hindi)
eng_tokenizer.fit_on_texts(english)

In [18]:
eng_tokens = eng_tokenizer.texts_to_sequences(english)
hin_tokens = eng_tokenizer.texts_to_sequences(hindi)

In [19]:
max_len = 0
for row in english:
    max_len = max(max_len, len(str(row).split()))


for row in hindi:
    max_len = max(max_len, len(str(row).split()))

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
eng_seq = pad_sequences(eng_tokens, padding = 'post', maxlen = max_len)
hin_seq = pad_sequences(hin_tokens, padding = 'post', maxlen = max_len)

In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [22]:
print(len(eng_tokenizer.word_index))
print(len(hin_tokenizer.word_index))

80060
100123


In [23]:
eng_vocab_size = 80061
hin_vocab_size = 100124

embed_size = 100

hidden_size = 64
lr = 0.01
epochs = 10
batch_size = 32

In [24]:
encoder = Encoder(eng_vocab_size, embed_size, hidden_size).to(device)
decoder = Decoder(hin_vocab_size, embed_size, hidden_size).to(device)

model = Seq2Seq(encoder, decoder).to(device)

compute_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

In [25]:
eng_tensor = torch.tensor(eng_seq, dtype = torch.long)
hin_tensor = torch.tensor(hin_seq, dtype = torch.long)

In [26]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(eng_tensor, hin_tensor)
dataloader = DataLoader(dataset, batch_size = batch_size, pin_memory = True, shuffle = True)

In [27]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for src, trg in dataloader:
        src = src.to(device)
        trg = trg.to(device)

        output = model(src, trg).to(device)

        output = output[:, 1:].reshape(-1, hin_vocab_size)

        trg = trg[:, 1:].reshape(-1)

        loss = compute_loss(output, trg)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch: {epoch+1}, Loss: {total_loss:.4f}')

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.98 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.22 GiB is free. Including non-PyTorch memory, this process has 10.34 GiB memory in use. Of the allocated memory 10.11 GiB is allocated by PyTorch, and 107.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)